# 05 - Google Cloud Cost and Performance Tradeoff Worksheet

This notebook mirrors notebook 04 for Google Cloud: **BigQuery vs Dataproc**.

## Goals
- Compare query-serving cost/performance (BigQuery) against batch distributed transforms (Dataproc)
- Quantify filtering and partition effects
- Produce a decision worksheet for architecture justification

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if Path.cwd().name == 'notebooks':
    REPO_ROOT = Path.cwd().parent
else:
    REPO_ROOT = Path.cwd()

OUT_DIR = REPO_ROOT / 'notebooks' / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Repo root:', REPO_ROOT)
print('Output dir:', OUT_DIR)

## 1) Enter Observed or Assumed Metrics

Edit these placeholders using your own lab/project values.

In [ ]:
# BigQuery pricing assumption (on-demand): about $5 per TB processed
BIGQUERY_PRICE_PER_TB = 5.0

# Dataproc job and cluster assumptions (illustrative)
DATAPROC_CLUSTER_HOURLY_USD = 0.82
DATAPROC_JOB_MINUTES = 25

# BigQuery processed data volumes (GB)
BQ_PROCESSED_UNFILTERED_GB = 3.0
BQ_PROCESSED_FILTERED_GB = 0.12

# Runtime assumptions (seconds)
BQ_RUNTIME_UNFILTERED_SEC = 14
BQ_RUNTIME_FILTERED_SEC = 3
DATAPROC_RUNTIME_SEC = DATAPROC_JOB_MINUTES * 60

# Query frequency assumptions
BQ_QUERIES_PER_DAY_UNFILTERED = 20
BQ_QUERIES_PER_DAY_FILTERED = 120

In [ ]:
def bq_query_cost_usd(processed_gb: float, price_per_tb: float = BIGQUERY_PRICE_PER_TB) -> float:
    return (processed_gb / 1024.0) * price_per_tb


def dataproc_job_cost_usd(hourly_rate: float = DATAPROC_CLUSTER_HOURLY_USD, minutes: float = DATAPROC_JOB_MINUTES) -> float:
    return hourly_rate * (minutes / 60.0)


bq_unfiltered_cost = bq_query_cost_usd(BQ_PROCESSED_UNFILTERED_GB)
bq_filtered_cost = bq_query_cost_usd(BQ_PROCESSED_FILTERED_GB)
dataproc_cost = dataproc_job_cost_usd()

summary = pd.DataFrame([
    {
        'mode': 'BigQuery (unfiltered query)',
        'cost_usd': round(bq_unfiltered_cost, 4),
        'runtime_sec': BQ_RUNTIME_UNFILTERED_SEC,
    },
    {
        'mode': 'BigQuery (partition-filtered query)',
        'cost_usd': round(bq_filtered_cost, 4),
        'runtime_sec': BQ_RUNTIME_FILTERED_SEC,
    },
    {
        'mode': 'Dataproc batch job',
        'cost_usd': round(dataproc_cost, 4),
        'runtime_sec': DATAPROC_RUNTIME_SEC,
    },
])

summary

## 2) Daily Cost Projection

Estimate recurring daily spend for query-heavy vs batch-heavy workloads.

In [ ]:
daily_cost_bq_unfiltered = bq_unfiltered_cost * BQ_QUERIES_PER_DAY_UNFILTERED
daily_cost_bq_filtered = bq_filtered_cost * BQ_QUERIES_PER_DAY_FILTERED
daily_cost_dataproc_batch = dataproc_cost  # assume one batch run/day

projection = pd.DataFrame([
    {'workload': 'BigQuery unfiltered (daily)', 'daily_cost_usd': daily_cost_bq_unfiltered},
    {'workload': 'BigQuery filtered (daily)', 'daily_cost_usd': daily_cost_bq_filtered},
    {'workload': 'Dataproc batch (daily)', 'daily_cost_usd': daily_cost_dataproc_batch},
]).sort_values('daily_cost_usd', ascending=False)

projection['daily_cost_usd'] = projection['daily_cost_usd'].round(4)
projection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

summary.plot.bar(x='mode', y='cost_usd', ax=axes[0], legend=False, color=['#1E88E5', '#43A047', '#FB8C00'])
axes[0].set_title('Per-Operation Cost')
axes[0].set_ylabel('USD')
axes[0].tick_params(axis='x', labelrotation=20)

summary.plot.bar(x='mode', y='runtime_sec', ax=axes[1], legend=False, color=['#1E88E5', '#43A047', '#FB8C00'])
axes[1].set_title('Runtime Comparison')
axes[1].set_ylabel('Seconds')
axes[1].tick_params(axis='x', labelrotation=20)

plt.tight_layout()
plt.show()

## 3) Break-Even Query Frequency

How many BigQuery queries at average processed volume equal one Dataproc batch run cost?

In [ ]:
AVG_BQ_PROCESSED_GB_PER_QUERY = 0.20
per_query_cost = bq_query_cost_usd(AVG_BQ_PROCESSED_GB_PER_QUERY)

if per_query_cost > 0:
    break_even_queries = dataproc_cost / per_query_cost
else:
    break_even_queries = np.inf

print(f'Average BigQuery processed/query: {AVG_BQ_PROCESSED_GB_PER_QUERY:.3f} GB')
print(f'BigQuery cost/query: ${per_query_cost:.5f}')
print(f'Dataproc batch run cost: ${dataproc_cost:.4f}')
print(f'Break-even query count vs one Dataproc run: {break_even_queries:.1f} queries')

## 4) Decision Worksheet

Use this table for architecture argumentation in assignments or design reviews.

In [ ]:
decision_template = pd.DataFrame([
    {'workload_type': 'Ad-hoc analyst SQL', 'recommended': 'BigQuery', 'reason': 'Serverless SQL, low ops overhead, fast iteration'},
    {'workload_type': 'Large scheduled ETL/feature build', 'recommended': 'Dataproc', 'reason': 'Distributed Spark transforms and execution control'},
    {'workload_type': 'Interactive dashboard on curated tables', 'recommended': 'BigQuery', 'reason': 'Fast query serving with partition-aware processing'},
    {'workload_type': 'Complex transformation DAG with custom Spark logic', 'recommended': 'Dataproc', 'reason': 'More control over distributed processing behavior'},
])

decision_template

In [ ]:
summary_out = OUT_DIR / 'gcp_cost_tradeoff_summary.csv'
projection_out = OUT_DIR / 'gcp_cost_tradeoff_daily_projection.csv'
decision_out = OUT_DIR / 'gcp_cost_tradeoff_decision_template.csv'

summary.to_csv(summary_out, index=False)
projection.to_csv(projection_out, index=False)
decision_template.to_csv(decision_out, index=False)

print('Wrote:')
print('-', summary_out)
print('-', projection_out)
print('-', decision_out)

## Reflection Prompts
1. Which variable dominates BigQuery spend in your scenario: processed GB/query or query frequency?
2. What storage/layout strategy would reduce bytes processed most?
3. When is Dataproc justified even with higher per-run cost?
4. How would you communicate this tradeoff to non-technical stakeholders?